# v09 · Charset order, and three things v09 does not do

In [1]:
import importlib.util, os, sys, time, math, hashlib, json
from collections import Counter
PROJ = os.environ.get("PROJ", "/home/rendier/Projects/ThePlace")
NB = os.path.join(PROJ, "FourthAgePapers/HyperindexingSystem/notebooks")
CORPUS = open(os.path.join(NB, "corpus_sample.txt"), encoding="utf-8").read()   # fixed 4000-char sample, 97-symbol keyboard set
def load(rel, name="m"):
    """Load a source file from the repo by path, unmodified."""
    spec = importlib.util.spec_from_file_location(name, os.path.join(PROJ, rel))
    m = importlib.util.module_from_spec(spec); spec.loader.exec_module(m); return m
print("corpus:", len(CORPUS), "chars, sha256", hashlib.sha256(CORPUS.encode()).hexdigest()[:16])

corpus: 4000 chars, sha256 3105964b9180566a


In [2]:
import sys
sys.path.insert(0, os.path.join(PROJ, "PtolemyDesktop"))
from Callimachus.v09.core.hyperwebster import HyperWebster as HW9
from Callimachus.v09.core import charset as cs
from Callimachus.v09.core.lsh_datatype import LSH_Datatype
from Callimachus.v09.database.hyperdatabase import HyperDatabase
from Callimachus.v09.gallery import hypergallery as hg9

## The charset order is the only variable, and the code carries three
Same 97 symbols, three orders: keyboard-row (v01–v07), Unicode codepoint (v09), and `a–z A–Z 0–9…` (v00 and the ValaQuenta module). One string, three integers.

In [3]:
kb = load("PtolemyDesktop/Callimachus/HyperWebster-Data-Storage/hyperwebster.py", "kb").HyperWebster()
vq = load("ValaQuenta/modules/hyperwebster/maths.py", "vq")
uni = HW9()
def addr_vq(s):                                # bijective form, ValaQuenta order, for comparison
    N = 97; a = 0
    for c in s: a = a*N + vq.US_KEYBOARD_CHARS.index(c) + 1
    return a - 1
for w in ("hello", "Ptolemy"):
    print(w, "| keyboard-row:", kb.point_to_text(w), "| Unicode:", uni.address_int(w), "| a-z first:", addr_vq(w))
assert len({kb.point_to_text("hello"), uni.address_int("hello"), addr_vq("hello")}) == 3
# the permutation between two of the orders
sig = [cs.PUBLIC_CHARSET.index(c) for c in kb.characters]
seen, cyc = set(), []
for i in range(97):
    if i in seen: continue
    j, n = i, 0
    while j not in seen: seen.add(j); j = sig[j]; n += 1
    cyc.append(n)
print("keyboard-row -> Unicode permutation, cycle type:", sorted(cyc, reverse=True), "| fixed points:", cyc.count(1), "| order:", math.lcm(*cyc))

hello | keyboard-row: 2937323952 | Unicode: 6706159586 | a-z first: 712911699
Ptolemy | keyboard-row: 60972185018825 | Unicode: 43236003041246 | a-z first: 35157909951779
keyboard-row -> Unicode permutation, cycle type: [92, 3, 2] | fixed points: 0 | order: 276


## `_pixels_to_int` carries a dead stub loop; the image cache still keys on a prefix

In [4]:
import inspect
src = inspect.getsource(hg9.HyperGallery._pixels_to_int)
print("dead stub present ('if False'):", "if False" in src)
s = hg9.ImageSpec(8, 8, hg9.PixelMode.GRAY8); G = hg9.HyperGallery(s)
a = [0]*64; b = [0]*64; b[63] = 200
ra, rb = G.index_image(a), G.index_image(b)
print("first-32-values cache: labels equal for different images:", ra.label == rb.label, "| regenerate(b) == b:", G.regenerate_image(rb) == b)

dead stub present ('if False'): True
first-32-values cache: labels equal for different images: True | regenerate(b) == b: False


## The SQLite 'Rabies' trigger never fires
It is declared `BEFORE UPDATE OF first_encountered ON words`; the table has no such column, and SQLite accepts the declaration without complaint.

In [5]:
import sqlite3
c = sqlite3.connect(":memory:"); c.executescript(__import__("Callimachus.v09.database.hyperdatabase", fromlist=["_SCHEMA"])._SCHEMA)
print("columns:", [r[1] for r in c.execute("PRAGMA table_info(words)")])
c.execute("INSERT INTO words VALUES('a','p',1,'t0',NULL,0)")
c.execute("UPDATE words SET indexed_at='changed' WHERE label='a'")
print("indexed_at after UPDATE:", c.execute("SELECT indexed_at FROM words").fetchone()[0])
assert c.execute("SELECT indexed_at FROM words").fetchone()[0] == "changed"

columns: ['label', 'payload', 'length', 'indexed_at', 'metadata', 'incomplete']
indexed_at after UPDATE: changed


`HyperDatabase.put` avoids touching `indexed_at` on conflict, so in practice the timestamp is kept by the SQL in `put`, not by the trigger.